# VaR Backtesting Example

This notebook demonstrates how to backtest VaR models.

In [ ]:
import sys
sys.path.append('..')

from src.data.data_collector import DataCollector
from src.data.preprocessor import DataPreprocessor
from src.models.var_calculator import VaRCalculator
from src.services.backtest_service import VaRBacktester
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

## 1. Prepare Data

In [ ]:
# Fetch data
collector = DataCollector()
ticker = 'AAPL'
data = collector.fetch_stock_data(ticker, period='2y')

# Calculate returns
preprocessor = DataPreprocessor()
returns_data = preprocessor.prepare_returns_data(data)
returns = returns_data['Returns'].dropna()

# Split into train/test
train_size = int(len(returns) * 0.7)
train_returns = returns.iloc[:train_size]
test_returns = returns.iloc[train_size:]

print(f"Training samples: {len(train_returns)}")
print(f"Testing samples: {len(test_returns)}")

## 2. Calculate VaR on Training Set

In [ ]:
# Calculate VaR
var_calc = VaRCalculator(confidence_level=0.95)
position_value = 1_000_000

var_result = var_calc.historical_var(train_returns, position_value)
print(f"VaR: ${var_result['VaR']:,.2f}")

# Create VaR estimates for test period
var_estimates = pd.Series(var_result['VaR'], index=test_returns.index)

## 3. Perform Backtesting

In [ ]:
# Run backtest
backtester = VaRBacktester(confidence_level=0.95)
results = backtester.backtest_var_model(test_returns, var_estimates, position_value)

print("\nBacktest Results:")
print(f"Observations: {results['num_observations']}")
print(f"Exceptions: {results['num_exceptions']}")
print(f"Exception Rate: {results['exception_rate']:.2%}")
print(f"Expected Rate: {results['expected_rate']:.2%}")
print(f"\nKupiec Test: {results['kupiec_test']['Result']}")
print(f"Traffic Light: {results['traffic_light_test']['Zone']}")

## 4. Visualize Exceptions

In [ ]:
# Plot actual losses vs VaR
actual_losses = -test_returns * position_value

plt.figure(figsize=(14, 6))
plt.plot(actual_losses.index, actual_losses.values, label='Actual Losses', alpha=0.7)
plt.axhline(y=var_result['VaR'], color='r', linestyle='--', label=f"VaR ({0.95:.0%})")
plt.fill_between(actual_losses.index, var_result['VaR'], actual_losses.max(), 
                 where=actual_losses > var_result['VaR'], color='red', alpha=0.3, label='Exceptions')
plt.title('VaR Backtesting: Actual Losses vs VaR Estimate')
plt.ylabel('Loss (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()